# Бейзлайны на ручных признаках

Сравню YetiRank, MultiClass и простые ориентиры на одной validation.
Здесь используется прежний восстановленный `start_ts`, в котором позже
найден канал утечки. Основной результат — в [итоговом сравнении](final-comparison.ipynb).

`report` читает сохранённые результаты; `recompute` готовит данные,
обучает модели и читает метрики нового запуска.


Сравню YetiRank, MultiClass и простые эвристики на одной validation.
В первом опыте использовал до 600 деревьев, early stopping 50 и seed 42.
Группа объединяет события пользователя за день. Это оценка ранжирования
наблюдавшихся событий, а не рекомендаций из всего каталога.

In [1]:
from pathlib import Path
import json
import os
import sys

import pandas as pd
from IPython.display import display

ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents)
            if (path / 'task_1/src').is_dir() and (path / 'docs/results').is_dir())
TASK = ROOT / 'task_1'
sys.path.insert(0, str(TASK / 'src'))
from notebook_workflow import from_environment

MODE = os.getenv('TLAB_NOTEBOOK_MODE', 'report')  # report или recompute
RUN_NAME = os.getenv('TLAB_NOTEBOOK_RUN', 'repeat_01')
DOWNLOAD_DATA = os.getenv('TLAB_NOTEBOOK_DOWNLOAD', '1') == '1'
INCLUDE_TEST = os.getenv('TLAB_INCLUDE_TEST', '0') == '1'
workflow = from_environment(MODE, RUN_NAME, DOWNLOAD_DATA, INCLUDE_TEST)

workflow.ensure('baseline')
BASELINE = workflow.path('baseline')
TUNING = workflow.path('optuna')
config = json.loads((BASELINE / 'run_config.json').read_text())
assert config['evaluation_split'] == 'val'
assert config['test_used'] is False


## Строки и признаки

In [2]:
display(pd.DataFrame.from_dict(config['split_stats'], orient='index'))
display(pd.Series(config['features'], name='Входной признак').to_frame())
print('Входов:', len(config['features']))

,rows_total,groups_total,rows_used,groups_used
train,8523909,187271,7931435,155600
val,3091912,68362,2882983,56946


,Входной признак
0,is_organic
1,track_length_seconds
2,ui_lst_amt_7
3,ui_skip_cnt_7
4,ui_lst_amt_14
5,ui_skip_cnt_14
6,ui_lst_amt_30
7,ui_skip_cnt_30
8,ui_lst_amt_90
9,ui_skip_cnt_90


Входов: 42


В исходном расчёте после удаления групп с одинаковой меткой осталось
7 931 435 строк train и 2 882 983 строки validation. В модель входят два
контекстных поля и 40 исторических признаков Item, User–Item и User–Artist.


## Частоты классов и простые ориентиры

In [3]:
display(pd.Series(config['class_prior'], name='Частота в train').to_frame())
metrics = pd.read_csv(BASELINE / 'validation_metrics.csv', float_precision='round_trip')
display(metrics.sort_values('NDCG@10', ascending=False).reset_index(drop=True).round(6))

,Частота в train
0,0.362909
1,0.629845
2,0.007246


,model,NDCG@10,QueryAUC,logloss
0,catboost YetiRank,0.768618,0.611703,NaN
1,"catboost MultiClass, E[label]",0.748124,0.593861,0.585730
2,user track decay 7d,0.720290,0.548940,NaN
3,user listened track 30d,0.716998,0.540689,NaN
4,user listened artist 30d,0.708656,0.543069,NaN
5,user listened track recently,0.694645,0.542804,NaN
6,user liked track,0.691493,0.514042,NaN
7,item popularity 7d,0.688738,0.522527,NaN
8,item popularity 30d,0.688537,0.523681,NaN
9,class prior,0.677148,0.500000,0.695037


Для MultiClass ранжирующий score равен `P(1) + 2 * P(2)`, а log loss
считается по всем трём вероятностям. У ранкера и счётчиков таких вероятностей
нет, поэтому log loss для них не указан. Частоты class prior считаются
только по train. В NDCG использована линейная релевантность 0/1/2.

## Сохранённые параметры и время

In [4]:
parameters = []
for name, record in config['models'].items():
    row = {'Модель': name}
    row.update({key: record['params'][key] for key in
                ['loss_function', 'depth', 'learning_rate', 'iterations', 'random_seed']})
    parameters.append(row)
display(pd.DataFrame(parameters))

timings = pd.read_csv(BASELINE / 'model_times.csv')
timings['fit_minutes'] = timings['fit_seconds'] / 60
display(timings[['model', 'mode', 'tree_count', 'training_threads', 'fit_minutes']].round(3))

,Модель,loss_function,depth,learning_rate,iterations,random_seed
0,YetiRank,YetiRank,6,0.1,600,42
1,MultiClass,MultiClass,6,0.1,600,42


,model,mode,tree_count,training_threads,fit_minutes
0,YetiRank,trained_with_validation,578,4,38.593
1,MultiClass,trained_with_validation,600,8,9.583


Время fit измеряется по готовым квантованным Pool.
YetiRank использует четыре потока, MultiClass — восемь.


## Какие признаки использовал ранкер

In [5]:
importance = pd.read_csv(BASELINE / 'feature_importance.csv')
display(importance.head(15).round(6))

,Feature Id,Importances_share_%,Importances_val_loss_change
0,ui_skip_cnt_90,12.560698,0.002811
1,ui_lst_decay_30,10.199812,0.001999
2,ua_skip_rate_90,9.666240,-0.000021
3,ui_lst_amt_90,6.597492,0.001461
4,ui_time_since_last_lst,6.406641,0.002252
5,ui_lst_decay_1,4.307231,0.000100
6,ui_skip_cnt_30,3.876565,-0.000070
7,ui_lst_decay_7,3.624321,0.000373
8,ua_lst_decay_1,3.606111,0.000581
9,item_avg_played_ratio_90,3.528744,-0.000077


PredictionValuesChange описывает изменение предсказаний,
LossFunctionChange — изменение функции потерь на validation.


## Результат исходного запуска

YetiRank получил NDCG@10 0.76862, MultiClass — 0.74812.
Затухающий счётчик User–Item за семь дней дал 0.72029.


## Расширенное обучение и Optuna

In [6]:
workflow.ensure('optuna')
tuned = pd.read_csv(TUNING / 'models.csv', float_precision='round_trip')
manual = tuned[tuned['method'] == 'manual']
columns = ['configuration', 'objective', 'NDCG@10', 'QueryAUC', 'logloss',
           'tree_count', 'fit_seconds']
display(manual[columns].round(6).reset_index(drop=True))


,configuration,objective,NDCG@10,QueryAUC,logloss,tree_count,fit_seconds
0,fixed_600,ranker,0.768618,0.611703,NaN,578,2315.550147
1,fixed_600,pointwise,0.748124,0.593861,0.585730,600,574.955164
2,control_1600,ranker,0.769794,0.612847,NaN,1574,5315.449479
3,control_1600,pointwise,0.749934,0.595595,0.584814,1600,1218.071395
4,tuned_1600,ranker,0.769328,0.612840,NaN,1549,5562.381027
5,tuned_1600,pointwise,0.749934,0.595595,0.584814,1600,1218.071395


Сравниваю исходные 600/50, контроль 1600/80 и Optuna при лимите 1600/80.
В исходном запуске лучшим ручным ранкером остался контроль: NDCG@10 0.769794.
Для MultiClass лучшими среди проверенных оказались исходные параметры.
Validation использовалась для подбора и early stopping.


## Продолжение

В режиме `recompute` Optuna также требует предварительного DFS130;
этот этап выполняется автоматически с тем же RUN_NAME.
Сравнение двух наборов — в [эксперименте 1](experiment-1.ipynb).
[Команды и ресурсы](../../docs/reproduction.md).
